# Try EMode now

[EMode](https://emodephotonix.com) is a mode solver for photonic waveguides. It finds the modes of a waveguide cross section (effective indices, losses and field profiles), and models how light travels through a device made of many sections.

This notebook runs EMode in your browser. You need no EMode account and nothing on your computer. The demo is the 2D solver, and each session lasts 60 minutes.

Run the cells in order, either one at a time with **Shift+Enter** or all at once with **Runtime > Run all**. The first cell sets up EMode, which takes about a minute.

> **EULA.** Running the setup cell installs EMode and means you accept the [EMode End User License Agreement](https://emodephotonix.com/legal/eula.pdf).

In [ ]:
#@title Set up EMode (about a minute)
# Running this cell installs EMode and accepts the EMode EULA (see above).
# It is safe to run again: what is already done is skipped.
import hashlib, importlib.metadata, json, os, platform, subprocess, sys, tempfile, urllib.error, urllib.request
from pathlib import Path

CREDENTIALS_URL = 'https://emodephotonix.com/wp-json/custom/v1/demo-credentials/'
EMODECONNECTION = '1.1.24'  # the client library this notebook is written for
MARKER = Path.home() / '.emode_demo_install.json'  # what this VM already installed


class DemoStop(Exception):
    """Stops the notebook with a message and no traceback."""
    def _render_traceback_(self):
        return [f'\033[1m{self}\033[0m']


class DemoPaused(DemoStop):
    """The demo is switched off for now."""


class CredentialsRejected(DemoStop):
    """The demo password changed while this cell ran."""


def fetch_credentials():
    """The demo account, the EMode build to install and where to check out a license."""
    request = urllib.request.Request(CREDENTIALS_URL, headers={'User-Agent': 'EMode-demo-notebook'})
    try:
        with urllib.request.urlopen(request, timeout=30) as reply:
            demo = json.load(reply)
    except urllib.error.HTTPError as error:
        if error.code == 429:
            raise DemoStop('This network has started the EMode demo too often in the last hour. '
                           'Please try again later.') from None
        raise DemoStop(f'The EMode demo service answered HTTP {error.code}. Please try again later.') from None
    except (urllib.error.URLError, TimeoutError) as error:
        raise DemoStop(f'Could not reach the EMode demo service ({error}). Check the connection and try again.') from None
    if not demo.get('enabled'):
        raise DemoPaused(demo.get('message') or 'The EMode demo is paused. Please try again later.')
    return demo


def install_client():
    try:
        installed = importlib.metadata.version('emodeconnection')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != EMODECONNECTION:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'emodeconnection=={EMODECONNECTION}'], check=True)


def install_emode(demo):
    """Download the demo's EMode build, check its SHA-256, install it, and return the executable."""
    from emodeconnection import installer
    wanted = {'version': demo['version'], 'sha256': demo['sha256']}
    if MARKER.exists():
        done = json.loads(MARKER.read_text())
        if {key: done.get(key) for key in wanted} == wanted and Path(done['exe']).exists():
            return Path(done['exe'])
    try:
        token = installer.login(demo['email'], demo['password'])
    except SystemExit:
        raise CredentialsRejected('The EMode download did not accept the demo account.') from None
    file_name = f"EMode-{demo['version']}-Linux-rslab.zip"
    with tempfile.TemporaryDirectory() as folder:
        try:
            zip_path = installer.download_release(token, 'linux-rslab', demo['version'], file_name, Path(folder))
        except SystemExit:
            raise DemoStop('The EMode download failed (see above). Please run this cell again.') from None
        digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
        if digest != demo['sha256']:
            raise DemoStop('The downloaded EMode does not match its published SHA-256, so it was not installed. '
                           'Please run this cell again, and tell hello@emodephotonix.com if it happens twice.')
        exe = installer.install_linux(zip_path, installer.get_default_install_dir('Linux'), explicit_dir=False)
    MARKER.write_text(json.dumps({**wanted, 'exe': str(exe)}))
    return exe


def log_in(exe, demo):
    """Store the demo account for EMode, and point it at the demo license manager."""
    secrets = {'EMODE_EMAIL': demo['email'], 'EMODE_PASSWORD': demo['password']}
    done = subprocess.run([str(exe), 'login', '--write'], env={**os.environ, **secrets},
                          stdin=subprocess.DEVNULL, capture_output=True, text=True)
    if done.returncode != 0:
        raise DemoStop(f'EMode could not store the demo login:\n{done.stdout}{done.stderr}')
    os.environ['EMODE_LICENSE_URL'] = demo['license_url']


def start(demo):
    import emodeconnection as emc
    from emodeconnection.types import EModeError, LicenseError
    exe = install_emode(demo)
    os.environ['PATH'] = f"{exe.parent}{os.pathsep}{os.environ['PATH']}"
    log_in(exe, demo)
    try:
        return emc.EMode()
    except LicenseError as error:
        if 'could not verify' in error.msg:
            raise CredentialsRejected(error.msg) from None
        raise DemoStop(error.msg) from None
    except EModeError as error:
        raise DemoStop(f'{error} Please run this cell again.') from None


if platform.system() != 'Linux':
    raise DemoStop('This demo runs on Linux, for example in Google Colab. To run EMode on your own '
                   'computer, get a free EMode2D license at https://emodephotonix.com/get-emode/')
if 'em' in globals() and getattr(em, 'running', False):
    em.close()  # a session from an earlier run of this cell

demo = fetch_credentials()
install_client()
try:
    em = start(demo)
except CredentialsRejected:
    # The demo password changes once a day. If that happened during this cell, fetch it again.
    print('The demo login was refused. Fetching it again and retrying once.')
    em = start(fetch_credentials())
del demo
print('EMode is ready.')

## 1. The modes of a silicon waveguide

A silicon core in silica, the workhorse of silicon photonics. All lengths are in nanometers. In Colab, move a slider and run the cell again to see how the modes change.

In [ ]:
width = 500  # @param {type:"slider", min:300, max:1000, step:20}
height = 220  # @param {type:"slider", min:150, max:400, step:10}
wavelength = 1550  # @param {type:"slider", min:1260, max:1650, step:10}

window_width, window_height = width + 2000, height + 2000
em.settings(
    wavelength=wavelength, x_resolution=10, y_resolution=10,
    window_width=window_width, window_height=window_height,
    num_modes=2, background_material='SiO2')
em.shape(name='core', material='Si', width=width, height=height, position=[0, window_height / 2])

em.FDM()     # solve for the modes
em.report()  # effective index, TE fraction and loss of each mode
em.plot()    # the first mode's field and the refractive index

## 2. Your own plots from the data

Every result is plain numpy, ready for your own analysis. Here are the electric field intensities of both modes, drawn with matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fields = em.get_fields(['Ex', 'Ey', 'Ez'])  # fields.field is a numpy array: (component, mode, x, y)
x, y = fields.grid.x, fields.grid.y         # the grid, in nm
n_eff = np.real(em.get('effective_index'))
intensity = np.sum(np.abs(fields.field) ** 2, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for mode, ax in enumerate(axes):
    ax.pcolormesh(x, y, intensity[mode].T, cmap='inferno', shading='auto')
    ax.add_patch(plt.Rectangle((-width / 2, (window_height - height) / 2), width, height, fill=False, color='white'))
    ax.set(title=f'Mode {mode}, n_eff = {n_eff[mode]:.4f}', xlabel='x (nm)', ylabel='y (nm)', aspect='equal')
plt.show()

## 3. Sweep the core width

Solve again at each width. A wider core holds the light more tightly, so the effective indices rise and more modes are guided.

In [ ]:
widths = np.arange(300, 1001, 50)  # nm
em.settings(num_modes=3)
result = em.sweep(key='shape, core, width', values=widths, result=['effective_index'])
em.shape(name='core', width=width)  # back to the width set in part 1

n_sweep = np.real(np.asarray(result['effective_index']))
n_clad = 1.444  # silica near 1550 nm. Below it, a mode is no longer guided.
plt.figure(figsize=(6, 4))
plt.plot(widths, np.where(n_sweep > n_clad, n_sweep, np.nan), 'o-')
plt.axhline(n_clad, color='gray', linestyle='--')
plt.xlabel('Core width (nm)')
plt.ylabel('Effective index')
plt.legend(['mode 0', 'mode 1', 'mode 2', 'silica cladding'])
plt.show()

## What next

This demo is EMode's 2D mode solver, for 60 minutes, on a shared demo account. EMode also models full devices with eigenmode expansion (EME), 3D structures, thermal and electro-optic effects, and nonlinear optics.

### Run more simulations now

Your session stays open until you run the last cell or the 60 minutes run out, so keep going:

- **Add a cell.** Click **+ Code** in the toolbar and work with `em`, the session the setup cell opened. Every EMode command runs on it, for example `em.settings(...)`, `em.shape(...)`, `em.FDM()`, `em.report()` and `em.plot()`.
- **Start over.** `em.reset('all')` clears the shapes and settings, so you can build a new structure in the same session.
- **Borrow from the examples.** The [examples in the documentation](https://docs.emodephotonix.com/examples.html) each come with a Python script. Copy the lines between `em = emc.EMode()` and `em.close()` into a new cell. Leave out those two lines: `em` already exists, and `em.close()` ends your demo session. The mode solver examples (for example SOI, SiN, LNOI and Dispersion) run in the demo. The EME examples need an EMode3D license.
- **Write your own.** Change a width, a material or a wavelength and solve again, or loop over a parameter with `em.sweep(...)` as in the cell above. The [function reference](https://docs.emodephotonix.com) lists every command and its options.

If a cell takes long, the demo is still running it. A message in the cell tells you when your 60 minutes are up.

### Keep using EMode

- **Get EMode2D free.** Create an account and get a free EMode2D license at [emodephotonix.com/get-emode](https://emodephotonix.com/get-emode/). An EMode3D trial is there too. Then `pip install emodeconnection` and `emode-install` on your own computer, where the same commands work after `em = emc.EMode()`.
- **Questions?** Write to [hello@emodephotonix.com](mailto:hello@emodephotonix.com).

When you are done, run the cell below to end your session.

In [ ]:
em.close()